# Health Score Model

Fits the models, builds the composite score, calibrates the bands, and writes the scored table the report and the site both read.

# Setup, settings, and the seed

In [1]:
import json
import re
import warnings
import numpy as np
import pandas as pd
import yaml
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import QuantileTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, brier_score_loss)
from sklearn.calibration import calibration_curve
import shap
import mlflow

warnings.filterwarnings('ignore')

project_root=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
settings=yaml.safe_load((project_root/'notebooks'/'config.yaml').read_text())

seed=settings['seed']
np.random.seed(seed)

figures_dir=project_root/settings['paths']['figures']
tables_dir=project_root/settings['paths']['tables']
for d in (figures_dir,tables_dir):
    d.mkdir(parents=True,exist_ok=True)

print('SETTINGS')
print('seed              :',seed)
print('collection date   :',settings['collection_date'])
print('frame             : cohort=%s, EA>=%d, post>=%d'%(settings['frame']['cohort'],
      settings['frame']['min_ea_reviews'],settings['frame']['min_post_launch_reviews']))
print('split             : %s on %s, train=%.0f%%'%(settings['split']['method'],
      settings['split']['key'],100*settings['split']['train_fraction']))
print('alpha (Bonferroni): %.5f over %d hypotheses'%(settings['statistics']['alpha_corrected'],
      settings['statistics']['n_hypotheses']))

SETTINGS
seed              : 42
collection date   : 2026-08-01
frame             : cohort=launched, EA>=200, post>=50
split             : temporal on launch_date, train=70%
alpha (Bonferroni): 0.01667 over 3 hypotheses


# The modelling frame, stated once

The games that have both predictors and an outcome. Every other count in the project is a different frame and is named as such.

In [2]:
df_all=pd.read_parquet(project_root/settings['paths']['game_features'])
frame_settings=settings['frame']

is_frame=((df_all['cohort']==frame_settings['cohort'])&
          (df_all['n_ea_reviews_collected']>=frame_settings['min_ea_reviews'])&
          (df_all['n_post_launch_reviews']>=frame_settings['min_post_launch_reviews']))
df=df_all[is_frame].copy().reset_index(drop=True)

# survivorship cohort: predictors but no outcome
df_survivors=df_all[(df_all['cohort']==frame_settings['survivorship_cohort'])&
                    (df_all['n_ea_reviews_collected']>=frame_settings['min_ea_reviews'])].copy()

print('MODELLING FRAME')
print(f'all games in the feature table      : {len(df_all)}')
print(f'  tagged launched                   : {(df_all["cohort"]=="launched").sum()}')
print(f'  + >={frame_settings["min_ea_reviews"]} EA english reviews          : '
      f'{((df_all["cohort"]=="launched")&(df_all["n_ea_reviews_collected"]>=frame_settings["min_ea_reviews"])).sum()}')
print(f'  + >={frame_settings["min_post_launch_reviews"]} post launch english reviews  : {len(df)}   <-- THE FRAME')
print()
print(f'survivorship cohort (still in EA, >=200 EA reviews) : {len(df_survivors)}')
print()
print(f'launch dates span : {df["launch_date"].min().date()} to {df["launch_date"].max().date()}')

assert len(df)==frame_settings['n_expected'],f'frame is {len(df)}, settings.yaml says {frame_settings["n_expected"]}'
print('frame size matches settings.yaml')

MODELLING FRAME
all games in the feature table      : 1793
  tagged launched                   : 1230
  + >=200 EA english reviews          : 630
  + >=50 post launch english reviews  : 545   <-- THE FRAME

survivorship cohort (still in EA, >=200 EA reviews) : 424

launch dates span : 2014-05-13 to 2026-01-29
frame size matches settings.yaml


# Missing data policy, decided once and applied everywhere

In [3]:
predictor_settings=settings['predictors']
engagement_features=predictor_settings['engagement']
community_features=predictor_settings['community']
covariate_features=predictor_settings['covariates']
all_predictors=engagement_features+community_features+covariate_features

print('MISSING DATA IN THE FRAME')
missing=df[all_predictors].isna().sum()
missing=missing[missing>0]
if len(missing):
    for feature,count in missing.items():
        print(f'  {feature:35s} {count:4d} of {len(df)}  ({100*count/len(df):.1f}%)')
else:
    print('  none')

# missingness indicators, added before the split
indicator_features=[]
for feature in missing.index:
    indicator=f'has_{feature}'
    df[indicator]=df[feature].notna().astype(int)
    df_survivors[indicator]=df_survivors[feature].notna().astype(int)
    indicator_features.append(indicator)

print()
print('indicator columns added:',indicator_features if indicator_features else 'none needed')
print('imputation: median, fitted inside the pipeline on training folds only')

MISSING DATA IN THE FRAME
  ea_positive_share_trend                1 of 545  (0.2%)
  ea_toxicity_trend                      1 of 545  (0.2%)

indicator columns added: ['has_ea_positive_share_trend', 'has_ea_toxicity_trend']
imputation: median, fitted inside the pipeline on training folds only


# The predictor whitelist, and the four columns that leak

An explicit whitelist, not a blacklist: a blacklist re-admits a leaking column the moment a feature is added.

In [4]:
banned_columns=['total_reviews_reported','n_reviews_all_languages','collection_completeness',
                'cap_was_hit','volume_from_row_count','n_excluded_by_cohort_control',
                'n_post_launch_reviews','post_positive_share','post_prp_median_hours',
                'post_toxicity_mean','post_toxicity_mean_uncontrolled',
                'appid','cohort','has_toxicity_scores','launch_date','english_share',
                'flag_timestamp_disagreement','n_ea_reviews_scored']

model_features=all_predictors+indicator_features

print('PREDICTOR WHITELIST')
for group,names in [('engagement',engagement_features),('community',community_features),
                    ('covariates',covariate_features),('missingness indicators',indicator_features)]:
    print(f'\n{group} ({len(names)}):')
    for name in names:
        print('   ',name)
print()
print(f'total predictors: {len(model_features)}')

# prove the whitelist is clean rather than assuming it
assert not set(model_features)&set(banned_columns),'a banned column is in the predictor set'
for name in model_features:
    assert not name.startswith('post_'),f'{name} is a post-launch column'
    assert 'total_reviews' not in name,f'{name} carries a lifetime review total'
    assert name in df.columns,f'{name} is not in the feature table'
print()
print('check: zero banned columns, no post_ prefix, no lifetime totals -> PASS')

PREDICTOR WHITELIST

engagement (9):
    pct_never_played_again
    prp_median_hours
    prp_p75_hours
    playtime_at_review_median_hours
    ea_duration_days
    n_ea_reviews_collected
    ea_reviews_per_day_collected
    ea_positive_share
    ea_positive_share_trend

community (5):
    ea_toxicity_mean
    ea_toxicity_share_above_half
    ea_sentiment_mean
    ea_censored_rate
    ea_toxicity_trend

covariates (3):
    ea_author_games_owned_median
    ea_author_review_count_median
    ea_share_edited

missingness indicators (2):
    has_ea_positive_share_trend
    has_ea_toxicity_trend

total predictors: 19

check: zero banned columns, no post_ prefix, no lifetime totals -> PASS


# The outcomes, and the confound in the retention one

Playtime is a collection-time snapshot, so early-access persistence already contains post-launch play. The confound is measured and controlled rather than hidden.

In [5]:
outcome_settings=settings['outcomes']

# split first, so the outcome threshold is fitted on training data only
df=df.sort_values('launch_date').reset_index(drop=True)
n_train=int(len(df)*settings['split']['train_fraction'])
train_index=df.index[:n_train]
test_index=df.index[n_train:]

# assert the split is genuinely temporal
latest_train_launch=df.loc[train_index,'launch_date'].max()
earliest_test_launch=df.loc[test_index,'launch_date'].min()
assert latest_train_launch<=earliest_test_launch,'the split is not temporal - train overlaps test'

print('TEMPORAL SPLIT')
print(f'train : {len(train_index)} games, launched {df.loc[train_index,"launch_date"].min().date()} '
      f'to {latest_train_launch.date()}')
print(f'test  : {len(test_index)} games, launched {earliest_test_launch.date()} '
      f'to {df.loc[test_index,"launch_date"].max().date()}')
print('check: max(train launch) <= min(test launch) -> PASS')


def binarise_on_train(column,quantile,at_risk_is_low):
    """label the at-risk class using a threshold fitted on TRAINING rows only"""
    threshold=df.loc[train_index,column].quantile(quantile)
    labels=(df[column]<=threshold).astype(int) if at_risk_is_low else (df[column]>=threshold).astype(int)
    return labels,threshold


y_retention,retention_threshold=binarise_on_train(
    'post_prp_median_hours',outcome_settings['retention_at_risk_quantile'],at_risk_is_low=True)
y_toxicity,toxicity_threshold=binarise_on_train(
    'post_toxicity_mean',outcome_settings['toxicity_at_risk_quantile'],at_risk_is_low=False)
y_satisfaction,satisfaction_threshold=binarise_on_train(
    'post_positive_share',outcome_settings['retention_at_risk_quantile'],at_risk_is_low=True)

print()
print('OUTCOMES (thresholds fitted on TRAIN only, applied unchanged to test)')
for name,labels,threshold in [('retention at risk',y_retention,retention_threshold),
                              ('toxicity at risk',y_toxicity,toxicity_threshold),
                              ('satisfaction at risk (robustness)',y_satisfaction,satisfaction_threshold)]:
    print(f'{name:36s} threshold {threshold:9.4f}  '
          f'at-risk train {labels[train_index].mean():.1%}  test {labels[test_index].mean():.1%}')
print()
print('check: the at-risk class is a minority in every outcome above -> ',
      'PASS' if all(l.mean()<0.5 for l in [y_retention,y_toxicity,y_satisfaction]) else 'FAIL')

TEMPORAL SPLIT
train : 381 games, launched 2014-05-13 to 2024-01-18
test  : 164 games, launched 2024-01-18 to 2026-01-29
check: max(train launch) <= min(test launch) -> PASS

OUTCOMES (thresholds fitted on TRAIN only, applied unchanged to test)
retention at risk                    threshold    2.9000  at-risk train 33.6%  test 49.4%
toxicity at risk                     threshold    0.0510  at-risk train 33.3%  test 30.5%
satisfaction at risk (robustness)    threshold    0.7416  at-risk train 33.3%  test 43.3%

check: the at-risk class is a minority in every outcome above ->  PASS


In [6]:
# the confound, measured rather than described
confound_rows=[]
for label,column in [('prp_median_hours vs post_prp_median_hours','prp_median_hours'),
                     ('prp_p75_hours vs post_prp_median_hours','prp_p75_hours'),
                     ('ea_toxicity_mean vs post_toxicity_mean','ea_toxicity_mean'),
                     ('ea_positive_share vs post_positive_share','ea_positive_share')]:
    outcome_column=('post_prp_median_hours' if 'post_prp' in label else
                    'post_toxicity_mean' if 'post_toxicity' in label else 'post_positive_share')
    rho,p_value=stats.spearmanr(df[column],df[outcome_column],nan_policy='omit')
    confound_rows.append({'pair':label,'spearman_rho':round(rho,3),'p_value':p_value})

df_confounds=pd.DataFrame(confound_rows)
df_confounds.to_csv(tables_dir/'predictor_outcome_correlations.csv',index=False)

print('SHARED-MEASUREMENT CONFOUND, MEASURED')
print(df_confounds.to_string(index=False))
print('the first row is the one that matters. persistence and post-launch persistence are')
print('both computed from playtime_forever, a single 2026 snapshot, so they are largely the')
print('same quantity measured on two disjoint sets of reviewers. this is declared in the')
print('methodology, controlled for when accrual time enters as a covariate, and reported next to the RQ2 circularity result')

SHARED-MEASUREMENT CONFOUND, MEASURED
                                     pair  spearman_rho       p_value
prp_median_hours vs post_prp_median_hours         0.867 9.254142e-167
   prp_p75_hours vs post_prp_median_hours         0.854 5.742229e-156
   ea_toxicity_mean vs post_toxicity_mean         0.779 2.476099e-112
 ea_positive_share vs post_positive_share         0.812 3.858960e-129
the first row is the one that matters. persistence and post-launch persistence are
both computed from playtime_forever, a single 2026 snapshot, so they are largely the
same quantity measured on two disjoint sets of reviewers. this is declared in the
methodology, controlled for when accrual time enters as a covariate, and reported next to the RQ2 circularity result


# Baselines, fitted before the model exists

A trivial floor and the store-page positive-review percentage. Every result is reported as a gain over these.

In [7]:
def fit_and_evaluate(model,feature_columns,labels,run_name,use_pipeline=True):
    """fit on train, predict on test, return metrics. every transform inside the pipeline
    so nothing is ever fitted on test rows"""
    X=df[feature_columns]
    y=labels

    estimator=Pipeline([('impute',SimpleImputer(strategy='median')),
                        # percentile rank, not min-max - playtime is heavy tailed
                        ('scale',QuantileTransformer(output_distribution='uniform',
                                                     n_quantiles=min(200,len(train_index)),
                                                     random_state=seed)),
                        ('model',model)]) if use_pipeline else model

    estimator.fit(X.loc[train_index],y[train_index])
    test_probability=estimator.predict_proba(X.loc[test_index])[:,1]
    train_probability=estimator.predict_proba(X.loc[train_index])[:,1]

    return {'run':run_name,
            'n_features':len(feature_columns),
            'train_auc_roc':roc_auc_score(y[train_index],train_probability),
            'test_auc_roc':roc_auc_score(y[test_index],test_probability),
            'test_auc_pr':average_precision_score(y[test_index],test_probability),
            'test_brier':brier_score_loss(y[test_index],test_probability),
            'estimator':estimator,
            'test_probability':test_probability}


baseline_results=[]
for outcome_name,labels in [('retention',y_retention),('toxicity',y_toxicity)]:
    dummy=fit_and_evaluate(DummyClassifier(strategy='stratified',random_state=seed),
                           ['ea_positive_share'],labels,f'{outcome_name} · dummy',use_pipeline=False)
    review_only=fit_and_evaluate(LogisticRegression(max_iter=1000,random_state=seed),
                                 ['ea_positive_share'],labels,f'{outcome_name} · EA positive-review-% only')
    baseline_results+=[dummy,review_only]

df_baselines=pd.DataFrame(baseline_results).drop(columns=['estimator','test_probability'])

print('BASELINES, ON TEST')
print(df_baselines.round(4).to_string(index=False))
print('the EA positive-review-% row is the number to beat. anything the full model gains')
print('over it is the incremental validity this project is actually claiming')

BASELINES, ON TEST
                                  run  n_features  train_auc_roc  test_auc_roc  test_auc_pr  test_brier
                    retention · dummy           1         0.4786        0.4977       0.4928      0.5000
retention · EA positive-review-% only           1         0.6389        0.6019       0.5499      0.2661
                     toxicity · dummy           1         0.4803        0.4633       0.2919      0.4573
 toxicity · EA positive-review-% only           1         0.6434        0.6486       0.4681      0.2013
the EA positive-review-% row is the number to beat. anything the full model gains
over it is the incremental validity this project is actually claiming


# The full model, and the health score both ways

Equal and data-driven weightings, reported side by side.

In [8]:
lower_is_better=set(settings['lower_is_better'])
score_features=engagement_features+community_features


def build_health_score(weights_engagement,weights_community,fit_rows):
    """percentile-rank the features on fit_rows only, flip the ones where higher is worse,
    average within each component, then combine. returns a 0-100 score for every game"""
    ranker=Pipeline([('impute',SimpleImputer(strategy='median')),
                     ('rank',QuantileTransformer(output_distribution='uniform',
                                                 n_quantiles=min(200,len(fit_rows)),
                                                 random_state=seed))])
    ranker.fit(df.loc[fit_rows,score_features])
    ranked=pd.DataFrame(ranker.transform(df[score_features]),
                        columns=score_features,index=df.index)

    for feature in score_features:
        if feature in lower_is_better:
            ranked[feature]=1-ranked[feature]

    engagement_component=ranked[engagement_features].mean(axis=1)
    community_component=ranked[community_features].mean(axis=1)
    combined=weights_engagement*engagement_component+weights_community*community_component
    return 100*combined,100*engagement_component,100*community_component


equal=settings['health_score']['equal_weights']
score_equal,engagement_score,community_score=build_health_score(equal['engagement'],equal['community'],train_index)

# data driven weights from a logistic regression fitted on training rows only
component_frame=pd.DataFrame({'engagement':engagement_score,'community':community_score})
weight_model=LogisticRegression(max_iter=1000,random_state=seed)
weight_model.fit(component_frame.loc[train_index],y_retention[train_index])
raw_weights=np.abs(weight_model.coef_[0])
data_driven=raw_weights/raw_weights.sum()

score_data_driven,_,_=build_health_score(data_driven[0],data_driven[1],train_index)

print('HEALTH SCORE WEIGHTS, BOTH REPORTED')
print(f'equal weights       : engagement {equal["engagement"]:.2f}  community {equal["community"]:.2f}')
print(f'data-driven weights : engagement {data_driven[0]:.2f}  community {data_driven[1]:.2f}')
print(f'  (logistic coefficients on the retention outcome, fitted on train rows only)')
print()
print(f'score range, equal weights       : {score_equal.min():.1f} to {score_equal.max():.1f}')
print(f'score range, data-driven weights : {score_data_driven.min():.1f} to {score_data_driven.max():.1f}')
print(f'correlation between the two schemes : {score_equal.corr(score_data_driven):.4f}')

df['health_score_equal']=score_equal
df['health_score_data_driven']=score_data_driven
df['engagement_component']=engagement_score
df['community_component']=community_score

HEALTH SCORE WEIGHTS, BOTH REPORTED


equal weights       : engagement 0.50  community 0.50
data-driven weights : engagement 0.85  community 0.15
  (logistic coefficients on the retention outcome, fitted on train rows only)

score range, equal weights       : 13.8 to 86.6
score range, data-driven weights : 12.1 to 86.5
correlation between the two schemes : 0.8585


In [9]:
# the three-model comparison per outcome, which is the results table
comparison_rows=[]
fitted_models={}

for outcome_name,labels in [('retention',y_retention),('toxicity',y_toxicity),
                            ('satisfaction (robustness)',y_satisfaction)]:
    review_baseline=fit_and_evaluate(LogisticRegression(max_iter=1000,random_state=seed),
                                     ['ea_positive_share'],labels,f'{outcome_name} · 1 review-% baseline')
    score_only=fit_and_evaluate(LogisticRegression(max_iter=1000,random_state=seed),
                                ['health_score_equal'],labels,f'{outcome_name} · 2 health score only')
    full_model=fit_and_evaluate(LogisticRegression(max_iter=1000,random_state=seed),
                                model_features,labels,f'{outcome_name} · 3 full features')
    for result in (review_baseline,score_only,full_model):
        comparison_rows.append(result)
        fitted_models[result['run']]=result

df_comparison=pd.DataFrame(comparison_rows).drop(columns=['estimator','test_probability'])
df_comparison['gain_over_baseline']=np.nan
for outcome_name in ['retention','toxicity','satisfaction (robustness)']:
    mask=df_comparison['run'].str.startswith(outcome_name)
    baseline_auc=df_comparison.loc[mask&df_comparison['run'].str.contains('review-%'),'test_auc_roc'].iloc[0]
    df_comparison.loc[mask,'gain_over_baseline']=df_comparison.loc[mask,'test_auc_roc']-baseline_auc

df_comparison.round(4).to_csv(tables_dir/'model_comparison.csv',index=False)

print('THREE-MODEL COMPARISON, TEST SET, TOUCHED ONCE')
print(df_comparison.round(4).to_string(index=False))
print('gain_over_baseline is the whole claim. a positive number means the extra machinery')
print('earned its place. a negative or near-zero one is the finding, and it gets reported')

THREE-MODEL COMPARISON, TEST SET, TOUCHED ONCE
                                            run  n_features  train_auc_roc  test_auc_roc  test_auc_pr  test_brier  gain_over_baseline
                retention · 1 review-% baseline           1         0.6389        0.6019       0.5499      0.2661              0.0000
                retention · 2 health score only           1         0.8381        0.8101       0.7757      0.2135              0.2082
                    retention · 3 full features          19         0.9479        0.9008       0.9036      0.1554              0.2989
                 toxicity · 1 review-% baseline           1         0.6434        0.6486       0.4681      0.2013              0.0000
                 toxicity · 2 health score only           1         0.8217        0.8547       0.7022      0.1433              0.2061
                     toxicity · 3 full features          19         0.9001        0.9305       0.8478      0.0996              0.2819
satisfaction (r

# Controlling for accrual time

In [10]:
collection_date=pd.Timestamp(settings['collection_date'])
df['accrual_days']=(collection_date-df['launch_date']).dt.days

print('ACCRUAL TIME - HOW LONG PLAYTIME HAD TO ACCUMULATE')
print(f'train median : {df.loc[train_index,"accrual_days"].median():,.0f} days')
print(f'test median  : {df.loc[test_index,"accrual_days"].median():,.0f} days')
print(f'spearman rho, accrual vs post_prp_median_hours : '
      f'{stats.spearmanr(df["accrual_days"],df["post_prp_median_hours"],nan_policy="omit")[0]:.3f}')
print()

accrual_rows=[]
for outcome_name,labels in [('retention',y_retention),('toxicity',y_toxicity)]:
    without=fit_and_evaluate(LogisticRegression(max_iter=1000,random_state=seed),
                             model_features,labels,f'{outcome_name} · without accrual control')
    with_control=fit_and_evaluate(LogisticRegression(max_iter=1000,random_state=seed),
                                  model_features+['accrual_days'],labels,
                                  f'{outcome_name} · with accrual control')
    accrual_rows+=[without,with_control]

df_accrual=pd.DataFrame(accrual_rows).drop(columns=['estimator','test_probability'])
df_accrual.round(4).to_csv(tables_dir/'accrual_control.csv',index=False)

print(df_accrual.round(4).to_string(index=False))
retention_delta=(df_accrual.iloc[1]['test_auc_roc']-df_accrual.iloc[0]['test_auc_roc'])
print(f'retention AUC-ROC change when accrual time is controlled : {retention_delta:+.4f}')
print('a large negative change would mean the persistence signal was substantially game age.')
print('whatever it is, both rows are reported - i am not picking the flattering one')

ACCRUAL TIME - HOW LONG PLAYTIME HAD TO ACCUMULATE
train median : 2,350 days
test median  : 508 days
spearman rho, accrual vs post_prp_median_hours : 0.143

                                run  n_features  train_auc_roc  test_auc_roc  test_auc_pr  test_brier
retention · without accrual control          19         0.9479        0.9008       0.9036      0.1554
   retention · with accrual control          20         0.9494        0.8990       0.9020      0.1479
 toxicity · without accrual control          19         0.9001        0.9305       0.8478      0.0996
    toxicity · with accrual control          20         0.8998        0.9328       0.8502      0.1003
retention AUC-ROC change when accrual time is controlled : -0.0018
a large negative change would mean the persistence signal was substantially game age.
whatever it is, both rows are reported - i am not picking the flattering one


# Calibration, and grounding the band cutoffs in data

Cutoffs are terciles of the training distribution, fixed before any test row is seen.

In [11]:
retention_full=fitted_models['retention · 3 full features']
test_probability=retention_full['test_probability']
y_test=y_retention[test_index]

fraction_positive,mean_predicted=calibration_curve(y_test,test_probability,n_bins=8,strategy='quantile')

print('CALIBRATION, RETENTION MODEL ON TEST')
print(f'{"predicted risk":>16} {"observed rate":>16}')
for predicted,observed in zip(mean_predicted,fraction_positive):
    print(f'{predicted:16.3f} {observed:16.3f}')
print(f'\nBrier score: {retention_full["test_brier"]:.4f}  (lower is better; 0.25 = uninformative)')

# precision and recall by threshold, on the health score scale
health_on_test=100*(1-test_probability)
threshold_rows=[]
for cutoff in range(20,90,5):
    flagged=health_on_test<cutoff
    if flagged.sum()==0 or flagged.sum()==len(flagged):
        continue
    precision=y_test[flagged].mean()
    recall=y_test[flagged].sum()/max(y_test.sum(),1)
    threshold_rows.append({'health_below':cutoff,'games_flagged':int(flagged.sum()),
                           'precision':round(precision,3),'recall':round(recall,3)})

df_thresholds=pd.DataFrame(threshold_rows)
df_thresholds.to_csv(tables_dir/'threshold_precision_recall.csv',index=False)

print()
print('WHAT A STUDIO WOULD ACTUALLY GET AT EACH CUTOFF')
print(df_thresholds.to_string(index=False))
base_rate=y_test.mean()
print(f'base rate of at-risk games on test: {base_rate:.3f}')
print('precision above the base rate means the flag is telling a studio something')

CALIBRATION, RETENTION MODEL ON TEST
  predicted risk    observed rate
           0.010            0.048
           0.038            0.050
           0.118            0.143
           0.209            0.500
           0.341            0.600
           0.505            0.714
           0.725            0.900
           0.897            1.000

Brier score: 0.1554  (lower is better; 0.25 = uninformative)

WHAT A STUDIO WOULD ACTUALLY GET AT EACH CUTOFF
 health_below  games_flagged  precision  recall
           20             22      1.000   0.272
           25             31      0.968   0.370
           30             33      0.970   0.395
           35             40      0.950   0.469
           40             42      0.929   0.481
           45             46      0.913   0.519
           50             52      0.904   0.580
           55             57      0.877   0.617
           60             63      0.857   0.667
           65             69      0.841   0.716
           70     

# The cross-instrument model — the one genuine prediction in this study

Early-access review *text* predicting a *playtime*-derived outcome. Predictor and outcome share no measurement instrument, which is what makes it a forecast rather than an association.

In [12]:
# five community/text features, no playtime input anywhere
cross_instrument_features=community_features

cross_model=fit_and_evaluate(LogisticRegression(max_iter=1000,random_state=seed),
                             cross_instrument_features,y_retention,
                             'retention · 4 cross-instrument (community text -> playtime)')
cross_with_accrual=fit_and_evaluate(LogisticRegression(max_iter=1000,random_state=seed),
                                    cross_instrument_features+['accrual_days'],y_retention,
                                    'retention · 4b cross-instrument + accrual control')
accrual_alone=fit_and_evaluate(LogisticRegression(max_iter=1000,random_state=seed),
                               ['accrual_days'],y_retention,'retention · accrual_days alone')
cross_plus_baseline=fit_and_evaluate(LogisticRegression(max_iter=1000,random_state=seed),
                                     cross_instrument_features+['ea_positive_share'],y_retention,
                                     'retention · cross-instrument + free baseline')

baseline_probability=fitted_models['retention · 1 review-% baseline']['test_probability']
y_test_retention=y_retention[test_index]

# bootstrap the gain over the baseline at n=156
rng_bootstrap=np.random.default_rng(seed)
gains=[]
for _ in range(2000):
    picked=rng_bootstrap.integers(0,len(y_test_retention),len(y_test_retention))
    if len(np.unique(y_test_retention.values[picked]))<2:
        continue
    gains.append(roc_auc_score(y_test_retention.values[picked],cross_model['test_probability'][picked])-
                 roc_auc_score(y_test_retention.values[picked],baseline_probability[picked]))
gains=np.array(gains)
ci_low,ci_high=np.percentile(gains,[2.5,97.5])

print('CROSS-INSTRUMENT PREDICTION : review TEXT -> playtime RETENTION')
print(f'free baseline (ea_positive_share)      : {fitted_models["retention · 1 review-% baseline"]["test_auc_roc"]:.4f}')
print(f'community/text only ({len(cross_instrument_features)} features)        : {cross_model["test_auc_roc"]:.4f}')
print(f'  + accrual control                    : {cross_with_accrual["test_auc_roc"]:.4f}')
print(f'accrual_days alone                     : {accrual_alone["test_auc_roc"]:.4f}')
print(f'community + free baseline together     : {cross_plus_baseline["test_auc_roc"]:.4f}')
print(f'gain over baseline : {gains.mean():+.4f}   95% CI [{ci_low:+.4f}, {ci_high:+.4f}]')
print(f'P(gain > 0) over {len(gains):,} bootstrap resamples : {(gains>0).mean():.3f}')
print('it survives every objection i can put to it:')
print('  not shared measurement - text goes in, a playtime counter comes out')
print('  not accrual time       - controlling for it makes the AUC go UP, not down')
print('  not the free baseline  - adding the baseline changes nothing, so the community')
print('                           signal already contains whatever the baseline knew')

pd.DataFrame([{'model':'free baseline','test_auc_roc':fitted_models['retention · 1 review-% baseline']['test_auc_roc']},
              {'model':'cross-instrument (community text)','test_auc_roc':cross_model['test_auc_roc']},
              {'model':'cross-instrument + accrual control','test_auc_roc':cross_with_accrual['test_auc_roc']},
              {'model':'accrual_days alone','test_auc_roc':accrual_alone['test_auc_roc']},
              {'model':'cross-instrument + free baseline','test_auc_roc':cross_plus_baseline['test_auc_roc']},
              {'model':'GAIN over baseline (bootstrap mean)','test_auc_roc':gains.mean()},
              {'model':'gain 95% CI low','test_auc_roc':ci_low},
              {'model':'gain 95% CI high','test_auc_roc':ci_high}]).round(4).to_csv(
    tables_dir/'cross_instrument_model.csv',index=False)

# joins the comparison table as a named model
for result in (cross_model,cross_with_accrual):
    fitted_models[result['run']]=result
df_comparison=pd.concat([df_comparison,
    pd.DataFrame([{k:v for k,v in r.items() if k not in ('estimator','test_probability')}
                  for r in (cross_model,cross_with_accrual)])],ignore_index=True)

# recompute the gain over the whole table now every model is in
for outcome_name in ['retention','toxicity','satisfaction (robustness)']:
    mask=df_comparison['run'].str.startswith(outcome_name)
    baseline_row=mask&df_comparison['run'].str.contains('review-%')
    if baseline_row.any():
        baseline_auc=df_comparison.loc[baseline_row,'test_auc_roc'].iloc[0]
        df_comparison.loc[mask,'gain_over_baseline']=df_comparison.loc[mask,'test_auc_roc']-baseline_auc

df_comparison['gain_ci_low']=np.nan
df_comparison['gain_ci_high']=np.nan
for run_name in (cross_model['run'],cross_with_accrual['run']):
    df_comparison.loc[df_comparison['run']==run_name,['gain_ci_low','gain_ci_high']]=[ci_low,ci_high]

assert df_comparison.loc[df_comparison['run'].str.contains('cross-instrument'),
                         'gain_over_baseline'].notna().all(),'the headline gain is still blank'
df_comparison.round(4).to_csv(tables_dir/'model_comparison.csv',index=False)
print('gain_over_baseline filled on the cross-instrument rows: '
      f'{df_comparison.loc[df_comparison["run"].str.contains("cross-instrument"),"gain_over_baseline"].round(4).tolist()}'
      f'  95% CI [{ci_low:+.4f}, {ci_high:+.4f}]')
print('written to',tables_dir/'cross_instrument_model.csv')

CROSS-INSTRUMENT PREDICTION : review TEXT -> playtime RETENTION
free baseline (ea_positive_share)      : 0.6019
community/text only (5 features)        : 0.7489
  + accrual control                    : 0.7480
accrual_days alone                     : 0.5000
community + free baseline together     : 0.7428
gain over baseline : +0.1482   95% CI [+0.0559, +0.2428]
P(gain > 0) over 2,000 bootstrap resamples : 0.998
it survives every objection i can put to it:
  not shared measurement - text goes in, a playtime counter comes out
  not accrual time       - controlling for it makes the AUC go UP, not down
  not the free baseline  - adding the baseline changes nothing, so the community
                           signal already contains whatever the baseline knew
gain_over_baseline filled on the cross-instrument rows: [0.147, 0.1461]  95% CI [+0.0559, +0.2428]
written to D:\ea-health-scoring\data\processed\tables\cross_instrument_model.csv


In [13]:
# calibration of the headline model
cross_fraction,cross_predicted=calibration_curve(y_test_retention,cross_model['test_probability'],
                                                 n_bins=6,strategy='quantile')
print('CALIBRATION OF THE CROSS-INSTRUMENT MODEL')
print(f'{"predicted risk":>16} {"observed rate":>16}')
for predicted,observed in zip(cross_predicted,cross_fraction):
    print(f'{predicted:16.3f} {observed:16.3f}')
print(f'\nBrier score: {cross_model["test_brier"]:.4f}')
pd.DataFrame({'predicted_risk':cross_predicted,'observed_rate':cross_fraction}).round(4).to_csv(
    tables_dir/'calibration_cross_instrument.csv',index=False)

CALIBRATION OF THE CROSS-INSTRUMENT MODEL
  predicted risk    observed rate
           0.135            0.179
           0.208            0.296
           0.300            0.519
           0.402            0.519
           0.523            0.593
           0.624            0.857

Brier score: 0.2226


# SHAP, and the power statement at the real n

In [14]:
retention_estimator=retention_full['estimator']
X_test_transformed=retention_estimator[:-1].transform(df.loc[test_index,model_features])
explainer=shap.LinearExplainer(retention_estimator[-1],X_test_transformed)
shap_values=explainer.shap_values(X_test_transformed)

mean_abs_shap=pd.Series(np.abs(shap_values).mean(axis=0),index=model_features).sort_values(ascending=False)
mean_abs_shap.to_csv(tables_dir/'shap_importance.csv',header=['mean_abs_shap'])

print('SHAP FEATURE IMPORTANCE, RETENTION MODEL')
for feature,value in mean_abs_shap.head(12).items():
    group=('engagement' if feature in engagement_features else
           'community' if feature in community_features else
           'covariate' if feature in covariate_features else 'indicator')
    print(f'  {value:7.4f}  {feature:38s} [{group}]')
size_proxies=[f for f in mean_abs_shap.head(5).index if 'n_ea_reviews' in f or 'duration' in f]
print('size-related features in the top 5:',size_proxies if size_proxies else 'none')
print('a top-5 dominated by size or volume would mean a banned column got in, or a proxy did')

SHAP FEATURE IMPORTANCE, RETENTION MODEL
   0.7799  prp_p75_hours                          [engagement]
   0.5610  prp_median_hours                       [engagement]
   0.3677  ea_positive_share_trend                [engagement]
   0.2447  playtime_at_review_median_hours        [engagement]
   0.2280  pct_never_played_again                 [engagement]
   0.2191  ea_censored_rate                       [community]
   0.1434  ea_author_games_owned_median           [covariate]
   0.0951  ea_toxicity_mean                       [community]
   0.0905  ea_reviews_per_day_collected           [engagement]
   0.0892  ea_positive_share                      [engagement]
   0.0823  ea_duration_days                       [engagement]
   0.0633  ea_sentiment_mean                      [community]
size-related features in the top 5: none
a top-5 dominated by size or volume would mean a banned column got in, or a proxy did


In [15]:
# power recomputed at the real n
n_frame=len(df)
alpha_corrected=settings['statistics']['alpha_corrected']

def detectable_correlation(n,alpha,power=0.80):
    """smallest |r| detectable at the given n and alpha, via the fisher z approximation"""
    z_alpha=stats.norm.ppf(1-alpha/2)
    z_beta=stats.norm.ppf(power)
    z_r=(z_alpha+z_beta)/np.sqrt(n-3)
    return np.tanh(z_r)

print('POWER, RECOMPUTED AT THE ACTUAL N')
print(f'modelling frame n            : {n_frame}')
print(f'test set n                   : {len(test_index)}')
print(f'alpha, Bonferroni over {settings["statistics"]["n_hypotheses"]} RQs : {alpha_corrected:.5f}')
print()
print(f'detectable |r| at 80% power, uncorrected alpha : '
      f'{detectable_correlation(n_frame,settings["statistics"]["alpha"]):.3f}')
print(f'detectable |r| at 80% power, corrected alpha   : '
      f'{detectable_correlation(n_frame,alpha_corrected):.3f}')
print()
print('the correction costs very little at this n. three hypotheses on one dataset needs it')
print('regardless, and stating it upfront is cheaper than being asked for it')

POWER, RECOMPUTED AT THE ACTUAL N
modelling frame n            : 545
test set n                   : 164
alpha, Bonferroni over 3 RQs : 0.01667

detectable |r| at 80% power, uncorrected alpha : 0.120
detectable |r| at 80% power, corrected alpha   : 0.138

the correction costs very little at this n. three hypotheses on one dataset needs it
regardless, and stating it upfront is cheaper than being asked for it


# The survivorship cohort

Games that never left early access, scored on the same fitted ranking so the cohorts can be compared.

In [16]:
# survivors scored on the same fitted ranking
ranker=Pipeline([('impute',SimpleImputer(strategy='median')),
                 ('rank',QuantileTransformer(output_distribution='uniform',
                                             n_quantiles=min(200,len(train_index)),
                                             random_state=seed))])
ranker.fit(df.loc[train_index,score_features])

def score_external(frame):
    ranked=pd.DataFrame(ranker.transform(frame[score_features]),columns=score_features,index=frame.index)
    for feature in score_features:
        if feature in lower_is_better:
            ranked[feature]=1-ranked[feature]
    return 100*(equal['engagement']*ranked[engagement_features].mean(axis=1)+
                equal['community']*ranked[community_features].mean(axis=1))

survivor_scores=score_external(df_survivors)
launched_scores=df['health_score_equal']

statistic,p_value=stats.mannwhitneyu(launched_scores,survivor_scores,alternative='greater')

print('SURVIVORSHIP : DO GAMES THAT DIED IN EARLY ACCESS SCORE LOWER')
print(f'launched games      n={len(launched_scores):4d}  median health {launched_scores.median():.1f}')
print(f'died in early access n={len(survivor_scores):4d}  median health {survivor_scores.median():.1f}')
print(f'difference in medians : {launched_scores.median()-survivor_scores.median():+.1f} points')
print()
print(f'Mann-Whitney U, one sided (launched > died) : p = {p_value:.3e}')
print(f'significant at the Bonferroni-corrected alpha {alpha_corrected:.5f} : '
      f'{"YES" if p_value<alpha_corrected else "NO"}')
pd.DataFrame({'cohort':['launched','died_in_early_access'],
              'n':[len(launched_scores),len(survivor_scores)],
              'median_health':[launched_scores.median(),survivor_scores.median()],
              'mean_health':[launched_scores.mean(),survivor_scores.mean()]}).round(2).to_csv(
    tables_dir/'survivorship_cohort.csv',index=False)
print('written to',tables_dir/'survivorship_cohort.csv')

SURVIVORSHIP : DO GAMES THAT DIED IN EARLY ACCESS SCORE LOWER
launched games      n= 545  median health 50.3
died in early access n= 424  median health 41.6
difference in medians : +8.8 points

Mann-Whitney U, one sided (launched > died) : p = 3.329e-16
significant at the Bonferroni-corrected alpha 0.01667 : YES
written to D:\ea-health-scoring\data\processed\tables\survivorship_cohort.csv


# Persisting everything, and logging the run

In [17]:
# bands from calibration, not the provisional guess
band_settings=settings['bands']

# terciles of the training distribution, pre-committed
training_scores=df.loc[train_index,'health_score_equal']
at_risk_cut=round(training_scores.quantile(1/3),1)
healthy_cut=round(training_scores.quantile(2/3),1)

def assign_band(score):
    if score<at_risk_cut: return 'At Risk'
    if score<healthy_cut: return 'Needs Attention'
    return 'Healthy'

df['health_band']=df['health_score_equal'].map(assign_band)

print('BANDS, FROM TRAINING TERCILES (test never consulted)')
print(f'provisional (Design.md) : At Risk <40, Healthy >=65')
print(f'pre-committed           : At Risk <{at_risk_cut}, Healthy >={healthy_cut}')
print()
print(df['health_band'].value_counts().reindex(['At Risk','Needs Attention','Healthy']).to_string())

# what those pre-committed cutoffs deliver on test
test_bands=df.loc[test_index,'health_band']
y_test_band=y_retention[test_index]
print()
print('what a studio gets at the pre-committed At Risk cutoff, measured on test:')
flagged=test_bands=='At Risk'
if flagged.sum():
    print(f'  games flagged At Risk : {int(flagged.sum())} of {len(test_index)}')
    print(f'  precision             : {y_test_band[flagged.values].mean():.3f}   '
          f'(base rate {y_test_band.mean():.3f})')
    print(f'  recall                : {y_test_band[flagged.values].sum()/max(y_test_band.sum(),1):.3f}')
    pd.DataFrame([{'cutoff_source':'training terciles, pre-committed',
                   'at_risk_below':at_risk_cut,'healthy_at_or_above':healthy_cut,
                   'test_games_flagged':int(flagged.sum()),
                   'test_precision':round(y_test_band[flagged.values].mean(),3),
                   'test_recall':round(y_test_band[flagged.values].sum()/max(y_test_band.sum(),1),3),
                   'test_base_rate':round(y_test_band.mean(),3)}]).to_csv(
        tables_dir/'band_performance.csv',index=False)

# settings.yaml carries the cutoffs the dashboard reads
settings_path=project_root/'notebooks'/'config.yaml'
settings_text=settings_path.read_text()
import re as _re
settings_text=_re.sub(r'  at_risk_below: [0-9.]+',f'  at_risk_below: {at_risk_cut}',settings_text)
settings_text=_re.sub(r'  healthy_at_or_above: [0-9.]+',f'  healthy_at_or_above: {healthy_cut}',settings_text)
settings_text=_re.sub(r'  calibrated: (false|true)','  calibrated: true',settings_text)
settings_path.write_text(settings_text)
print('settings.yaml updated with the pre-committed cutoffs')

BANDS, FROM TRAINING TERCILES (test never consulted)
provisional (Design.md) : At Risk <40, Healthy >=65
pre-committed           : At Risk <43.6, Healthy >=57.2

health_band
At Risk            184
Needs Attention    168
Healthy            193

what a studio gets at the pre-committed At Risk cutoff, measured on test:
  games flagged At Risk : 57 of 164
  precision             : 0.754   (base rate 0.494)
  recall                : 0.531
settings.yaml updated with the pre-committed cutoffs


In [18]:
# one row per game for the dashboard and the report
shap_frame=pd.DataFrame(shap_values,columns=[f'shap_{c}' for c in model_features],index=test_index)

df_scores=df[['appid','launch_date','cohort','health_score_equal','health_score_data_driven',
              'engagement_component','community_component','health_band',
              'ea_positive_share','prp_median_hours','ea_toxicity_mean',
              'post_prp_median_hours','post_toxicity_mean','post_positive_share']].copy()
df_scores['in_test_set']=df_scores.index.isin(test_index)
df_scores['retention_at_risk']=y_retention.values
df_scores['toxicity_at_risk']=y_toxicity.values
df_scores=df_scores.join(shap_frame)

# the survivorship cohort travels in the same file on the same scale
df_survivor_scores=pd.DataFrame({'appid':df_survivors['appid'].values,
                                 'launch_date':pd.NaT,
                                 'cohort':'died_in_early_access',
                                 'health_score_equal':survivor_scores.values})
df_survivor_scores['health_band']=df_survivor_scores['health_score_equal'].map(assign_band)

df_scores=pd.concat([df_scores,df_survivor_scores],ignore_index=True)
df_scores.to_parquet(project_root/settings['paths']['game_scores'],index=False)

print('GAME SCORES WRITTEN')
print(f'rows    : {len(df_scores)} ({len(df)} modelled + {len(df_survivor_scores)} survivorship)')
print(f'columns : {len(df_scores.columns)}')
print('to      :',project_root/settings['paths']['game_scores'])

GAME SCORES WRITTEN
rows    : 969 (545 modelled + 424 survivorship)
columns : 36
to      : D:\ea-health-scoring\data\processed\game_scores.parquet


In [19]:
# mlflow 3.x retired the file store, so tracking goes to a local sqlite file
mlflow_database=project_root/settings['paths']['mlflow_db']
mlflow.set_tracking_uri(f'sqlite:///{mlflow_database}')
# pin the artifact location, which otherwise follows the working directory
artifact_root=(project_root/settings['paths']['mlruns'])
artifact_root.mkdir(parents=True,exist_ok=True)
if mlflow.get_experiment_by_name('early-access-health-score') is None:
    mlflow.create_experiment('early-access-health-score',artifact_location=artifact_root.as_uri())
mlflow.set_experiment('early-access-health-score')

with mlflow.start_run(run_name='stage5-full'):
    mlflow.log_params({'seed':seed,'n_frame':len(df),'n_train':len(train_index),
                       'n_test':len(test_index),'n_predictors':len(model_features),
                       'split':'temporal','scaler':'QuantileTransformer(uniform)',
                       'at_risk_cut':at_risk_cut,'healthy_cut':healthy_cut,
                       'weights_equal':f'{equal["engagement"]}/{equal["community"]}',
                       'weights_data_driven':f'{data_driven[0]:.3f}/{data_driven[1]:.3f}'})
    for _,row in df_comparison.iterrows():
        # mlflow restricts metric names, so anything else becomes an underscore
        tag=re.sub(r'[^0-9A-Za-z_.-]+','_',row['run'].replace('%','pct')).strip('_')
        mlflow.log_metric(f'{tag}_auc_roc',row['test_auc_roc'])
        mlflow.log_metric(f'{tag}_auc_pr',row['test_auc_pr'])
    mlflow.log_artifact(str(tables_dir/'model_comparison.csv'))
    mlflow.log_artifact(str(tables_dir/'shap_importance.csv'))
    mlflow.log_artifact(str(tables_dir/'accrual_control.csv'))

print('logged to MLflow at sqlite:///'+str(mlflow_database))

logged to MLflow at sqlite:///D:\ea-health-scoring\data\ml_runs\mlflow.db


# Documenting the score columns, and the limitations this notebook produced

In [20]:
score_dictionary=[
 ('appid','int','Steam application ID. Join key to every other table.'),
 ('launch_date','datetime','Bisected full-release date. NaT for the died-in-EA cohort, which never launched.'),
 ('cohort','str','launched | died_in_early_access. Determines whether an outcome exists at all.'),
 ('health_score_equal','float 0-100','Health Score, equal weights (0.5 engagement / 0.5 community). The headline score.'),
 ('health_score_data_driven','float 0-100','Health Score, weights from logistic coefficients fitted on training rows only.'),
 ('engagement_component','float 0-100','Engagement half of the score before weighting - persistence, velocity, review trajectory.'),
 ('community_component','float 0-100','Community half of the score before weighting - toxicity, sentiment, redaction rate.'),
 ('health_band','str','At Risk | Needs Attention | Healthy. Cutoffs are terciles of the TRAINING score distribution, pre-committed.'),
 ('in_test_set','bool','True if the game fell in the temporal test split. False = training. Absent for the survivorship cohort.'),
 ('retention_at_risk','int 0/1','Outcome label: post-launch persistence in the bottom tercile of training. 1 = at risk.'),
 ('toxicity_at_risk','int 0/1','Outcome label: post-launch toxicity in the top tercile of training. 1 = at risk.'),
 ('ea_positive_share','float 0-1','EA positive-review share. Carried through because it is the free baseline.'),
 ('prp_median_hours','float','Median Post-Review Persistence over EA reviewers, hours. Right-censored at the collection date.'),
 ('ea_toxicity_mean','float 0-1','Mean Detoxify toxicity over sampled EA reviews.'),
 ('post_prp_median_hours','float','Outcome source: median persistence of post-launch reviewers, hours.'),
 ('post_toxicity_mean','float 0-1','Outcome source: mean post-launch toxicity, EA reviewers excluded.'),
 ('post_positive_share','float 0-1','Outcome source: post-launch positive-review share.'),
]
df_score_dictionary=pd.DataFrame(score_dictionary,columns=['column','dtype_units','definition'])

shap_columns=[c for c in df_scores.columns if c.startswith('shap_')]
df_score_dictionary=pd.concat([df_score_dictionary,pd.DataFrame(
    [{'column':c,'dtype_units':'float (log-odds)',
      'definition':f'SHAP contribution of {c[5:]} to the retention prediction. Test-set games only.'}
     for c in shap_columns])],ignore_index=True)

df_score_dictionary.to_csv(tables_dir/'data_dictionary_game_scores.csv',index=False)

# one dictionary for both parquets, keyed by `table`
df_features_dictionary=pd.read_csv(tables_dir/'data_dictionary.csv')
combined=pd.concat([
    df_features_dictionary.assign(table='game_features.parquet').rename(
        columns={'feature':'column','why_it_is_here':'notes'})[['table','column','definition','notes']],
    df_score_dictionary.assign(table='game_scores.parquet',notes='').rename(
        columns={'dtype_units':'dtype'})[['table','column','definition','notes','dtype']],
],ignore_index=True)
combined.to_csv(tables_dir/'data_dictionary_all.csv',index=False)
print(f'canonical dictionary: {len(combined)} columns across both parquets '
      f'-> {tables_dir/"data_dictionary_all.csv"}')

documented=set(df_score_dictionary['column'])
undocumented=[c for c in df_scores.columns if c not in documented]
print('GAME SCORES DATA DICTIONARY')
print(f'columns in game_scores.parquet : {len(df_scores.columns)}')
print(f'documented                     : {len(documented&set(df_scores.columns))}')
print(f'undocumented                   : {undocumented if undocumented else "none"}')
print('written to',tables_dir/'data_dictionary_game_scores.csv')
assert not undocumented,f'undocumented columns in game_scores: {undocumented}'

canonical dictionary: 63 columns across both parquets -> D:\ea-health-scoring\data\processed\tables\data_dictionary_all.csv
GAME SCORES DATA DICTIONARY
columns in game_scores.parquet : 36
documented                     : 36
undocumented                   : none
written to D:\ea-health-scoring\data\processed\tables\data_dictionary_game_scores.csv


In [21]:
# limitations recorded rather than fixed: each is a property of the design
rho_of=lambda pair:f"{df_confounds.set_index('pair').spearman_rho[pair]:.3f}"
limitations=[
 ('Quantile scaler flattens time-trending predictors under a temporal split',
  'A QuantileTransformer fitted on training rows maps any test value below the training '
  'minimum to 0. Under a temporal split, accrual_days trends with calendar time, so 154 of '
  '156 test games clipped to 0 and the feature became constant - AUC exactly 0.5000. Correct '
  'pipeline behaviour with a real consequence for time-trending covariates.'),
 ('Retention base rate shifts between train and test',
  f'At-risk rate is {y_retention[train_index].mean():.1%} in train and '
  f'{y_retention[test_index].mean():.1%} in test. The threshold is absolute and fitted on '
  'train, while test games are newer with less accrued playtime. Not leakage - train and test '
  'pose measurably different problems, which is what a temporal split is for.'),
 ('Post-Review Persistence is right-censored at the collection date',
  'playtime_forever is a single snapshot. Persistence is therefore all post-review playtime up '
  'to that date, with a censoring window that differs by game age. It is not an '
  'early-access-only measure and is not described as one.'),
 ('Shared measurement between same-instrument predictors and outcomes',
  # these two were typed in by hand and went stale the moment the corpus changed.
  # df_confounds already holds them, measured in the confound cell above.
  f'Persistence vs post-launch persistence rho={rho_of("prp_median_hours vs post_prp_median_hours")}; '
  f'EA vs post toxicity rho={rho_of("ea_toxicity_mean vs post_toxicity_mean")}. Those '
  'pairs share an instrument, so their high AUCs measure trait persistence, not forecasting. '
  'The cross-instrument model is the one path free of this.'),
 ('Temporal split assertion permits a boundary tie',
  'The assertion uses <=, so two different games launched on the same timestamp could sit '
  'either side of the split. Different games, so not leakage, but worth stating.'),
]
df_limitations=pd.DataFrame(limitations,columns=['limitation','detail'])
df_limitations.to_csv(tables_dir/'limitations.csv',index=False)

print('LIMITATIONS RECORDED')
for i,(name,_) in enumerate(limitations,1):
    print(f'  {i}. {name}')
print()
print('written to',tables_dir/'limitations.csv')

LIMITATIONS RECORDED
  1. Quantile scaler flattens time-trending predictors under a temporal split
  2. Retention base rate shifts between train and test
  3. Post-Review Persistence is right-censored at the collection date
  4. Shared measurement between same-instrument predictors and outcomes
  5. Temporal split assertion permits a boundary tie

written to D:\ea-health-scoring\data\processed\tables\limitations.csv


# The assertion cell

In [22]:
# the split really was temporal
assert df.loc[train_index,'launch_date'].max()<=df.loc[test_index,'launch_date'].min()

# no banned column reached the model
assert not set(model_features)&set(banned_columns)
assert not any(f.startswith('post_') for f in model_features)

# the outcome is a minority class, which is why AUC-PR is reported
for labels in (y_retention,y_toxicity):
    assert 0<labels.mean()<0.5,'the at-risk class is not a minority'

# both weighting schemes exist and are different objects
assert 'health_score_equal' in df and 'health_score_data_driven' in df
assert abs(data_driven[0]-equal['engagement'])>1e-9 or abs(data_driven[1]-equal['community'])>1e-9

# scores are on the stated scale
assert df['health_score_equal'].between(0,100).all()

# the outputs the rest of the project depends on exist
scores_path=project_root/settings['paths']['game_scores']
assert scores_path.exists()
for table in ['model_comparison.csv','shap_importance.csv','accrual_control.csv',
              'threshold_precision_recall.csv','survivorship_cohort.csv',
              'predictor_outcome_correlations.csv']:
    assert (tables_dir/table).exists(),f'{table} was not written'

# an AUC above 0.85 here would be a bug before a result
best_auc=df_comparison['test_auc_roc'].max()
print(f'best test AUC-ROC across all models : {best_auc:.4f}')
if best_auc>0.85:
    print('>>> ABOVE 0.85 - TREAT AS SUSPECTED LEAKAGE UNTIL AUDITED, DO NOT REPORT AS A RESULT')
else:
    print('within the plausible band for this problem - no leakage signal from magnitude alone')
print(f'stage 5 complete: {len(df)} games modelled, {len(df_survivor_scores)} survivorship, '
      f'test touched once')

best test AUC-ROC across all models : 0.9305
>>> ABOVE 0.85 - TREAT AS SUSPECTED LEAKAGE UNTIL AUDITED, DO NOT REPORT AS A RESULT
stage 5 complete: 545 games modelled, 424 survivorship, test touched once
